In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# mypy: disable-error-code="import-not-found"

# The notebook should be executed from the project root directory
import os
import sys
from pathlib import Path

if "_correct_path" not in locals():
    os.chdir("..")
    sys.path.append(".")
    print(f"changed dir to {Path('.').resolve()})")
    _correct_path = True

In [ ]:
import os

import pandas as pd

from utils.schema import DatasetInput

In [ ]:
dataset_url = "https://s3.amazonaws.com/datarobot_public_datasets/10k_diabetes_20.csv"

df = pd.read_csv(dataset_url)
# Replace non-JSON compliant values
# df = df.replace([float("inf"), -float("inf")], None)  # Replace infinity with None
# df = df.where(pd.notnull(df), None)  # Replace NaN with None

# Create dataset dictionary
dataset = DatasetInput(
    name=os.path.splitext(os.path.basename(dataset_url))[0],
    data=df.to_dict("records"),
)

In [ ]:
from utils.api import cleanse_dataframes

cleansed_data = await cleanse_dataframes([dataset])

In [ ]:
from utils.api import get_dictionary

dictionary = await get_dictionary([dataset])

In [ ]:
dictionary.dictionaries[0].dictionary[:5]

In [ ]:
from utils.api import rephrase_message
from utils.schema import ChatRequest

question = "What are some interesting insights about the medication?"
chat_response = await rephrase_message(
    messages=ChatRequest(
        messages=[
            {
                "role": "user",
                "content": "What are some interesting insights about the medication?",
            },
            {
                "role": "user",
                "content": "Show me a nice plot about it",
            },
        ],
    )
)

In [ ]:
from utils.api import run_analysis
from utils.schema import RunAnalysisRequest

analysis_request = RunAnalysisRequest(
    data={ds.name: ds.data for ds in cleansed_data.datasets},
    dictionary={
        d["name"]: d["dictionary"] for d in dictionary.model_dump()["dictionaries"]
    },
    question=chat_response["enhanced_user_message"],
)
analysis_result = await run_analysis(analysis_request)

In [ ]:
import asyncio

from utils.api import get_business_analysis, run_charts
from utils.schema import BusinessAnalysisRequest, RunChartsRequest

chart_df = (
    pd.DataFrame(analysis_result.data)
    if isinstance(analysis_result.data, list)
    else pd.DataFrame([analysis_result.data])
)

# Prepare requests
chart_request = RunChartsRequest(
    data=chart_df.to_dict("records"),
    question=chat_response.get("enhanced_question", question),
)

business_request = BusinessAnalysisRequest(
    data=chart_df.to_dict("records"),
    dictionary=[
        {
            "column": col,
            "description": "Analysis result column",
            "data_type": str(chart_df[col].dtype),
        }
        for col in chart_df.columns
    ],
    question=chat_response.get("enhanced_question", question),
)

# Create and start tasks immediately
charts_task = asyncio.create_task(run_charts(chart_request))
business_task = asyncio.create_task(get_business_analysis(business_request))

In [ ]:
import plotly.offline as pyo

from utils.schema import BusinessAnalysisResult, RunChartsResult

pyo.init_notebook_mode()

tasks = [charts_task, business_task]

# Wait for each task to complete
for coro in asyncio.as_completed(tasks):
    result = await coro

    # Determine which task completed by checking the result structure
    if isinstance(result, RunChartsResult) and (result.fig1 or result.fig2):
        if result.fig1:
            pyo.iplot(result.fig1)
        if result.fig2:
            pyo.iplot(result.fig2)

    elif isinstance(result, BusinessAnalysisResult):
        print(f"Bottom Line:\n{(result.bottom_line or '')}")

        print(f"Additional Insights:\n{result.additional_insights}")

        print("Follow-up Questions:")
        for q in result.follow_up_questions:
            print(f"- {q}")

In [ ]:
with open("tests/models/run_analysis_result.json", "w") as f:
    f.write(analysis_result.model_dump_json(indent=4))
with open("tests/models/run_charts_result.json", "w") as f:
    f.write(charts_task.result().model_dump_json(indent=4))
with open("tests/models/run_business_result.json", "w") as f:
    f.write(business_task.result().model_dump_json(indent=4))